# NFL Play-by-Play Data Exploration

This notebook explores the play-by-play database schema and demonstrates the rich data available for analysis.

## Goal
Show MS students what's available in the database to inspire contributions and research projects.

## Data Source
- **nflverse** play-by-play data (1999-2025)
- 372+ fields including advanced analytics (EPA, WPA, CPOE)
- FTN charting data (2022+) - formations, personnel
- Snap counts (2012+) - player participation

In [1]:
# Import required libraries
import sys
from pathlib import Path

import pandas as pd

# Add src to path
src_path = Path.cwd().parent.parent / "src"
sys.path.insert(0, str(src_path))

from ffpy.database import FFPyDatabase

# Connect to database
db = FFPyDatabase()
print(f"Database: {db.db_path}")
print("Connected successfully!")

2025-12-22 10:06:37.214 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Database: C:\Users\ricka\.ffpy\ffpy.db
Connected successfully!


## 1. Database Schema Overview

Let's explore what tables and columns are available.

In [2]:
# Get all tables
tables_query = """
    SELECT name FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
"""
tables = pd.read_sql(tables_query, db.conn)
print("Available tables:")
print(tables)

Available tables:
                 name
0        actual_stats
1        api_requests
2          data_loads
3        ftn_charting
4               games
5   player_id_mapping
6             players
7               plays
8         projections
9         snap_counts
10    sqlite_sequence


In [3]:
# Get schema for plays table
plays_schema = pd.read_sql("PRAGMA table_info(plays)", db.conn)
print(f"\nPlays table has {len(plays_schema)} columns:")
print(plays_schema[["cid", "name", "type"]].to_string(index=False))


Plays table has 173 columns:
 cid                          name      type
   0                       play_id      TEXT
   1                       game_id      TEXT
   2                   old_game_id      TEXT
   3                        season   INTEGER
   4                   season_type      TEXT
   5                          week   INTEGER
   6                     game_date      TEXT
   7                           qtr   INTEGER
   8     quarter_seconds_remaining   INTEGER
   9        game_seconds_remaining   INTEGER
  10        half_seconds_remaining   INTEGER
  11                     game_half      TEXT
  12                         drive   INTEGER
  13                       posteam      TEXT
  14                       defteam      TEXT
  15                     home_team      TEXT
  16                     away_team      TEXT
  17                 side_of_field      TEXT
  18                          down   INTEGER
  19                       ydstogo   INTEGER
  20                  yar

In [4]:
# Get schema for games table
games_schema = pd.read_sql("PRAGMA table_info(games)", db.conn)
print(f"\nGames table has {len(games_schema)} columns:")
print(games_schema[["cid", "name", "type"]].to_string(index=False))


Games table has 21 columns:
 cid          name      type
   0       game_id      TEXT
   1   old_game_id      TEXT
   2        season   INTEGER
   3   season_type      TEXT
   4          week   INTEGER
   5     game_date      TEXT
   6     home_team      TEXT
   7     away_team      TEXT
   8    home_score   INTEGER
   9    away_score   INTEGER
  10          roof      TEXT
  11       surface      TEXT
  12          temp   INTEGER
  13          wind   INTEGER
  14   spread_line      REAL
  15    total_line      REAL
  16      location      TEXT
  17       stadium      TEXT
  18 game_finished   INTEGER
  19    created_at TIMESTAMP
  20    updated_at TIMESTAMP


## 2. Data Statistics

How much data do we have?

In [5]:
# Count records
stats_query = """
    SELECT
        (SELECT COUNT(*) FROM plays) as total_plays,
        (SELECT COUNT(*) FROM games) as total_games,
        (SELECT COUNT(DISTINCT season) FROM plays) as seasons_loaded,
        (SELECT MIN(season) FROM plays) as earliest_season,
        (SELECT MAX(season) FROM plays) as latest_season
"""
stats = pd.read_sql(stats_query, db.conn)
print("\nDatabase Statistics:")
print(f"  Total Plays: {stats['total_plays'].iloc[0]:,}")
print(f"  Total Games: {stats['total_games'].iloc[0]:,}")
print(f"  Seasons: {stats['earliest_season'].iloc[0]} - {stats['latest_season'].iloc[0]}")
print(f"  Seasons Loaded: {stats['seasons_loaded'].iloc[0]}")


Database Statistics:
  Total Plays: 5,089
  Total Games: 1,093
  Seasons: 2022 - 2025
  Seasons Loaded: 4


In [6]:
# Plays by season
by_season = pd.read_sql(
    """
    SELECT
        season,
        COUNT(*) as plays,
        COUNT(DISTINCT game_id) as games,
        COUNT(DISTINCT week) as weeks
    FROM plays
    GROUP BY season
    ORDER BY season DESC
""",
    db.conn,
)

print("\nPlays by Season:")
print(by_season.to_string(index=False))


Plays by Season:
 season  plays  games  weeks
   2025    162     29     12
   2024   4167    136     19
   2023     66     15     11
   2022    694     69     13


## 3. Advanced Metrics Available

The database includes cutting-edge analytics metrics.

In [7]:
# Sample advanced metrics from plays table
advanced_metrics = pd.read_sql(
    """
    SELECT
        play_id,
        desc,
        epa,           -- Expected Points Added
        wpa,           -- Win Probability Added
        cpoe,          -- Completion % Over Expected
        air_epa,       -- EPA from air yards
        yac_epa,       -- EPA from yards after catch
        comp_air_epa,  -- Completed air EPA
        xyac_epa       -- Expected YAC EPA
    FROM plays
    WHERE play_type = 'pass'
        AND epa IS NOT NULL
    LIMIT 10
""",
    db.conn,
)

print("\nSample Advanced Metrics (passing plays):")
print(advanced_metrics)


Sample Advanced Metrics (passing plays):
  play_id                                               desc       epa  \
0    83.0  (14:27) 1-K.Murray pass short left to 6-J.Conn...  2.028874   
1   108.0  (13:43) (Shotgun) 1-K.Murray pass short middle...  0.754242   
2   199.0  (11:08) (Shotgun) 1-K.Murray pass short middle...  1.680800   
3   224.0  (10:32) (Shotgun) 1-K.Murray pass incomplete s... -0.467625   
4   347.0  (8:34) (Shotgun) 1-K.Murray sacked ob at BUF 5... -0.985448   
5   381.0  (7:54) (Shotgun) 1-K.Murray pass short left to...  2.704705   
6   503.0  (6:15) (Shotgun) 17-J.Allen pass short middle ...  0.913284   
7   528.0  (5:33) (Shotgun) 17-J.Allen sacked at ARI 30 f... -6.044650   
8   587.0  (4:46) (Shotgun) 1-K.Murray pass short left to... -0.547628   
9   622.0  (4:11) (Shotgun) 1-K.Murray pass short right t...  2.664915   

        wpa       cpoe   air_epa   yac_epa  comp_air_epa  xyac_epa  
0  0.053842  13.140899 -1.083852  3.112726     -1.083852  1.345418  
1  0.

## 4. Player Performance Examples

Let's look at some interesting player queries.

In [8]:
# Top QBs by EPA in 2024 (minimum 100 dropbacks)
top_qbs = pd.read_sql(
    """
    SELECT
        passer_player_name as player,
        COUNT(*) as dropbacks,
        ROUND(AVG(epa), 3) as avg_epa,
        ROUND(SUM(epa), 2) as total_epa,
        SUM(complete_pass) as completions,
        SUM(pass_touchdown) as tds,
        SUM(interception) as ints
    FROM plays
    WHERE season = 2024
        AND qb_dropback = 1
        AND passer_player_name IS NOT NULL
    GROUP BY passer_player_name
    HAVING dropbacks >= 100
    ORDER BY avg_epa DESC
    LIMIT 10
""",
    db.conn,
)

print("\nTop 10 QBs by EPA/play (2024):")
print(top_qbs.to_string(index=False))


Top 10 QBs by EPA/play (2024):
Empty DataFrame
Columns: [player, dropbacks, avg_epa, total_epa, completions, tds, ints]
Index: []


In [9]:
# Top receivers by target share (2024)
top_receivers = pd.read_sql(
    """
    WITH receiver_targets AS (
        SELECT
            receiver_player_name,
            posteam,
            COUNT(*) as targets,
            SUM(complete_pass) as receptions,
            SUM(yards_gained) as yards,
            ROUND(AVG(air_yards), 1) as avg_air_yards,
            SUM(touchdown) as tds
        FROM plays
        WHERE season = 2024
            AND play_type = 'pass'
            AND receiver_player_name IS NOT NULL
        GROUP BY receiver_player_name, posteam
    ),
    team_targets AS (
        SELECT
            posteam,
            COUNT(*) as team_targets
        FROM plays
        WHERE season = 2024
            AND play_type = 'pass'
        GROUP BY posteam
    )
    SELECT
        r.receiver_player_name as player,
        r.posteam as team,
        r.targets,
        r.receptions,
        r.yards,
        r.tds,
        ROUND(100.0 * r.targets / t.team_targets, 1) as target_share_pct,
        r.avg_air_yards
    FROM receiver_targets r
    JOIN team_targets t ON r.posteam = t.posteam
    WHERE r.targets >= 40
    ORDER BY target_share_pct DESC
    LIMIT 15
""",
    db.conn,
)

print("\nTop 15 Receivers by Target Share (2024):")
print(top_receivers.to_string(index=False))


Top 15 Receivers by Target Share (2024):
Empty DataFrame
Columns: [player, team, targets, receptions, yards, tds, target_share_pct, avg_air_yards]
Index: []


## 5. Situational Analysis

The database enables rich situational queries.

In [12]:
# Red Zone efficiency by team (2024)
redzone = pd.read_sql(
    """
    SELECT
        posteam as team,
        COUNT(*) as plays,
        SUM(CASE WHEN touchdown = 1 THEN 1 ELSE 0 END) as tds,
        ROUND(100.0 * SUM(touchdown) / COUNT(*), 1) as td_rate,
        ROUND(AVG(epa), 3) as avg_epa,
        SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) as passes,
        SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) as runs
    FROM plays
    WHERE yardline_100 <= 20
        AND season = 2024
        AND play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY td_rate DESC
""",
    db.conn,
)

print("\nRed Zone Efficiency by Team (2024):")
print(redzone.to_string(index=False))


Red Zone Efficiency by Team (2024):
team  plays  tds  td_rate  avg_epa  passes  runs
  NO     13    5     38.5    0.258       6     7
  TB     15    5     33.3    0.345       4    11
 DAL      9    3     33.3   -0.084       7     2
 NYJ     10    3     30.0    0.521       6     4
 IND     14    4     28.6   -0.189       3    11
 BUF     25    6     24.0    0.010       9    16
 MIN     14    3     21.4   -0.308       7     7
  KC     15    3     20.0   -0.044       5    10
 BAL     16    3     18.8   -0.029       8     8
 WAS     11    2     18.2    0.230       6     5
 DET     19    3     15.8    0.023      10     9
 JAX     13    2     15.4   -0.594       6     7
 DEN     13    2     15.4   -0.072       7     6
 CAR     13    2     15.4   -0.487       8     5
 ATL     13    2     15.4   -0.029       8     5
  LA     16    2     12.5   -0.459       7     9
 ARI     16    2     12.5   -0.327       6    10
  SF     17    2     11.8   -0.341       7    10
 CIN     17    2     11.8   -0.2

In [13]:
# Third down conversion rates (2024)
third_downs = pd.read_sql(
    """
    SELECT
        posteam as team,
        COUNT(*) as attempts,
        SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) as conversions,
        ROUND(100.0 * SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) as conversion_rate,
        ROUND(AVG(ydstogo), 1) as avg_distance,
        ROUND(AVG(epa), 3) as avg_epa
    FROM plays
    WHERE down = 3
        AND season = 2024
        AND play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY conversion_rate DESC
""",
    db.conn,
)

print("\nThird Down Conversion Rates (2024):")
print(third_downs.to_string(index=False))

DatabaseError: Execution failed on sql '
    SELECT 
        posteam as team,
        COUNT(*) as attempts,
        SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) as conversions,
        ROUND(100.0 * SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) as conversion_rate,
        ROUND(AVG(ydstogo), 1) as avg_distance,
        ROUND(AVG(epa), 3) as avg_epa
    FROM plays
    WHERE down = 3
        AND season = 2024
        AND play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY conversion_rate DESC
': no such column: first_down

## 6. Game Context Fields

Rich situational context is available for every play.

In [14]:
# Sample play showing available context
context_example = pd.read_sql(
    """
    SELECT
        game_id,
        game_date,
        week,
        posteam,
        defteam,
        down,
        ydstogo,
        yardline_100,
        quarter_seconds_remaining,
        half_seconds_remaining,
        game_seconds_remaining,
        score_differential,
        posteam_timeouts_remaining,
        defteam_timeouts_remaining,
        wp,              -- Win probability
        def_wp,          -- Defensive win probability
        home_wp,         -- Home team win probability
        away_wp,         -- Away team win probability
        desc
    FROM plays
    WHERE touchdown = 1
        AND season = 2024
    LIMIT 5
""",
    db.conn,
)

print("\nSample Plays with Context (Touchdowns):")
print(context_example.T)  # Transpose for readability


Sample Plays with Context (Touchdowns):
                                                                            0  \
game_id                                                       2024_01_ARI_BUF   
game_date                                                          2024-09-08   
week                                                                        1   
posteam                                                                   ARI   
defteam                                                                   BUF   
down                                                                        3   
ydstogo                                                                     5   
yardline_100                                                                5   
quarter_seconds_remaining                                                 474   
half_seconds_remaining                                                   1374   
game_seconds_remaining                                              

## 7. Player Identification Fields

Players are tracked with consistent IDs and positions.

In [15]:
# Sample player ID fields
player_ids = pd.read_sql(
    """
    SELECT DISTINCT
        passer_player_id,
        passer_player_name,
        receiver_player_id,
        receiver_player_name,
        rusher_player_id,
        rusher_player_name
    FROM plays
    WHERE season = 2024
        AND (passer_player_name IS NOT NULL OR rusher_player_name IS NOT NULL)
    LIMIT 10
""",
    db.conn,
)

print("\nPlayer ID Fields Example:")
print(player_ids)


Player ID Fields Example:
  passer_player_id passer_player_name receiver_player_id receiver_player_name  \
0             None               None               None                 None   
1       00-0035228           K.Murray         00-0033553             J.Conner   
2       00-0035228           K.Murray         00-0035500             G.Dortch   
3       00-0035228           K.Murray         00-0039849           M.Harrison   
4             None               None               None                 None   
5       00-0035228           K.Murray               None                 None   
6       00-0035228           K.Murray         00-0038559            Mi.Wilson   
7             None               None               None                 None   
8       00-0034857            J.Allen         00-0033555            M.Hollins   
9       00-0034857            J.Allen               None                 None   

  rusher_player_id rusher_player_name  
0       00-0033553           J.Conner  
1

## 8. Research Ideas for MS Students

This database enables many interesting research projects:

### Machine Learning / Predictive Analytics
1. **Play Prediction**: Predict play type (run/pass) based on down, distance, score, time
2. **EPA Prediction**: Build models to predict EPA before the play
3. **Win Probability**: Improve win probability models using play-level features
4. **Player Performance**: Predict future player performance based on historical trends
5. **Injury Risk**: Analyze play characteristics and snap counts for injury prediction

### Statistical Analysis
6. **Situational Tendencies**: What plays work best in different situations?
7. **Coaching Decisions**: Analyze 4th down decisions, 2-point conversions
8. **Home Field Advantage**: Quantify home field impact by venue, team, weather
9. **Player Clustering**: Group players by playing style using advanced metrics
10. **Team Strategy Evolution**: How have team strategies changed over time?

### Optimization
11. **Play Calling Optimization**: Optimal play mix given situation
12. **Personnel Packages**: Which formations/personnel are most effective?
13. **Target Distribution**: Optimal target share for receivers
14. **Timeout Management**: When should teams use timeouts?

### Fantasy Football
15. **Projection Models**: Better player projections using play-level data
16. **Matchup Analysis**: Predict player performance vs specific defenses
17. **Usage Patterns**: Snap counts, target share, red zone opportunities
18. **DFS Optimization**: Daily fantasy lineup optimization with variance modeling

### Visualization
19. **Interactive Dashboards**: Team/player performance dashboards
20. **Play Visualization**: Animate plays or create play diagrams
21. **Trend Analysis**: Show how metrics evolve throughout season/career

## 9. Getting Started with Your Own Analysis

Here's a template for your own queries:

In [16]:
# Template: Customize this for your analysis
your_query = """
    SELECT
        -- Choose your fields
        season,
        week,
        posteam,
        play_type,
        yards_gained,
        epa
    FROM plays
    WHERE 1=1
        -- Add your filters
        AND season = 2024
        -- AND week = 1
        -- AND posteam = 'KC'
        -- AND play_type = 'pass'
    LIMIT 100
"""

# Run your query
results = pd.read_sql(your_query, db.conn)
print("\nYour Analysis Results:")
print(results.head(10))


Your Analysis Results:
   season  week posteam play_type  yards_gained       epa
0    2024     1    None      None           NaN  0.000000
1    2024     1     ARI   kickoff           0.0  0.257819
2    2024     1     ARI       run           3.0 -0.200602
3    2024     1     ARI      pass          22.0  2.028874
4    2024     1     ARI      pass           9.0  0.754242
5    2024     1     ARI       run           2.0 -0.029602
6    2024     1     ARI       run           2.0 -0.247749
7    2024     1     ARI       run           2.0 -0.530139
8    2024     1     ARI      pass           8.0  1.680800
9    2024     1     ARI      pass           0.0 -0.467625


## 10. Next Steps

1. **Explore the schema**: Run `PRAGMA table_info(plays)` to see all 372+ fields
2. **Try example queries**: Modify the queries above for different teams/players/seasons
3. **Join with other data**: Combine with rosters, betting lines, weather data
4. **Build models**: Export to CSV and use your favorite ML library
5. **Create visualizations**: Use plotly, matplotlib, or seaborn
6. **Contribute back**: Add new features to FFPy!

### Resources
- [nflverse Documentation](https://nflverse.nflverse.com/)
- [FFPy Database Guide](../../docs/db/DATABASE_GUIDE.md)
- [Play Analysis Example](../../examples/play_analysis_example.py)

In [17]:
# Close database connection
db.close()
print("Database connection closed.")

Database connection closed.
